# Module 4c: Breast Cancer Recurrence Prediction (Neural Network)

## Aim
Build and tune a neural-network model to predict breast cancer recurrence (`recurrence-events`) from the provided dataset.

## Why this matters
In this setting, **false negatives** (predicting no recurrence when recurrence will happen) are especially costly because they could delay closer monitoring or treatment decisions.


In [ ]:
# Colab setup for legacy .xls reading
!pip -q install xlrd==2.0.1


In [ ]:
import os
import random
from dataclasses import dataclass
from datetime import date, datetime
from typing import Dict, List, Tuple

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_SEED = 42
TARGET_COL = "Class"
POSITIVE_CLASS = "recurrence-events"

def set_seed(seed=RANDOM_SEED):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)

set_seed()


## 1) Load and clean the data

The original UCI-style dataset has some categorical ranges and unknown markers (`?`).

Cleaning steps:
1. Load `.xls` file.
2. Ensure target column name is `Class`.
3. Normalize date-like Excel parsing issues in range columns.
4. Replace `?` with `Unknown` so rows are retained.


In [ ]:
KNOWN_RANGES: Dict[str, set[str]] = {
    "tumor-size": {
        "0-4", "5-9", "10-14", "15-19", "20-24", "25-29",
        "30-34", "35-39", "40-44", "45-49", "50-54"
    },
    "inv-nodes": {
        "0-2", "3-5", "6-8", "9-11", "12-14", "15-17", "18-20", "21-23", "24-26"
    },
}

def normalize_excel_range(value, column_name):
    if isinstance(value, str):
        return value.strip()
    if isinstance(value, (datetime, date, pd.Timestamp)):
        dt = pd.to_datetime(value)
        candidates = [f"{dt.day}-{dt.month}", f"{dt.month}-{dt.year % 100}"]
        allowed = KNOWN_RANGES.get(column_name, set())
        for c in candidates:
            if c in allowed:
                return c
        return candidates[0]
    return str(value).strip()

def load_and_clean_data(path):
    df = pd.read_excel(path, engine="xlrd")
    if TARGET_COL not in df.columns:
        df.columns = [*df.columns[:-1], TARGET_COL]

    for col in ("tumor-size", "inv-nodes"):
        if col in df.columns:
            df[col] = df[col].apply(lambda x: normalize_excel_range(x, col))

    for col in df.columns:
        if df[col].dtype == object:
            df[col] = df[col].astype(str).str.strip().replace({"?": "Unknown"})
    return df

df = load_and_clean_data("breast-cancer.xls")
df.head()


## 2) Quick exploratory checks

This checks dataset size and class balance. If the positive class is smaller, accuracy alone is not enough; recall, PR-AUC, and confusion matrix become important.


In [ ]:
print("Shape:", df.shape)
print("\nClass distribution:")
print(df[TARGET_COL].value_counts())
print("\nClass proportions:")
print(df[TARGET_COL].value_counts(normalize=True).round(3))

plt.figure(figsize=(6, 4))
sns.countplot(data=df, x=TARGET_COL)
plt.title("Target Class Distribution")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()


## 3) Preprocessing and train/test split

Pipeline design:
1. Binary target: recurrence vs non-recurrence.
2. Stratified train/test split (80/20).
3. One-hot encode categorical predictors.
4. Standardize numeric predictor (`deg-malig`).


In [ ]:
y = (df[TARGET_COL] == POSITIVE_CLASS).astype(int).values
X = df.drop(columns=[TARGET_COL]).copy()

numeric_cols = [c for c in X.columns if c == "deg-malig"]
categorical_cols = [c for c in X.columns if c not in numeric_cols]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_SEED
)

def build_preprocessor(categorical_cols, numeric_cols):
    return ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols),
            ("num", StandardScaler(), numeric_cols),
        ],
        remainder="drop",
    )


## 4) Step-by-step model tuning (5-fold stratified CV)

I compare several neural-network architectures and hyperparameters.

Selection rule:
1. Highest ROC-AUC
2. If tied, highest PR-AUC
3. If still tied, highest F1

I also tune the decision threshold on each fold to maximize F1 (instead of forcing 0.50).


In [ ]:
@dataclass
class ModelConfig:
    hidden_units: Tuple[int, ...]
    alpha: float
    learning_rate_init: float
    max_iter: int

def build_model(config):
    return MLPClassifier(
        hidden_layer_sizes=config.hidden_units,
        activation="relu",
        solver="adam",
        alpha=config.alpha,
        learning_rate_init=config.learning_rate_init,
        batch_size=16,
        max_iter=config.max_iter,
        early_stopping=True,
        n_iter_no_change=20,
        random_state=RANDOM_SEED,
    )

def best_threshold_for_f1(y_true, y_prob):
    thresholds = np.linspace(0.1, 0.9, 81)
    scores = [f1_score(y_true, (y_prob >= t).astype(int), zero_division=0) for t in thresholds]
    return float(thresholds[int(np.argmax(scores))])

def evaluate_config_cv(X, y, categorical_cols, numeric_cols, config, n_splits=5):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_SEED)

    auc_scores, ap_scores, f1_scores, thresholds = [], [], [], []

    for tr_idx, va_idx in skf.split(X, y):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]

        pre = build_preprocessor(categorical_cols, numeric_cols)
        X_tr_p = pre.fit_transform(X_tr).astype(np.float32)
        X_va_p = pre.transform(X_va).astype(np.float32)

        model = build_model(config)
        model.fit(X_tr_p, y_tr)
        y_prob = model.predict_proba(X_va_p)[:, 1]

        th = best_threshold_for_f1(y_va, y_prob)
        y_pred = (y_prob >= th).astype(int)

        auc_scores.append(roc_auc_score(y_va, y_prob))
        ap_scores.append(average_precision_score(y_va, y_prob))
        f1_scores.append(f1_score(y_va, y_pred, zero_division=0))
        thresholds.append(th)

    return {
        "auc_mean": float(np.mean(auc_scores)),
        "auc_std": float(np.std(auc_scores)),
        "ap_mean": float(np.mean(ap_scores)),
        "f1_mean": float(np.mean(f1_scores)),
        "best_threshold": float(np.mean(thresholds)),
    }

configs = [
    ModelConfig((64, 32), 1e-4, 1e-3, 1200),
    ModelConfig((128, 64), 5e-4, 8e-4, 1400),
    ModelConfig((128, 64, 32), 1e-4, 8e-4, 1500),
    ModelConfig((96, 48), 1e-3, 5e-4, 1200),
    ModelConfig((64, 64, 32), 1e-5, 6e-4, 1300),
    ModelConfig((128, 32), 1e-4, 1e-3, 1200),
]

rows = []
for i, cfg in enumerate(configs, 1):
    res = evaluate_config_cv(X_train, y_train, categorical_cols, numeric_cols, cfg, n_splits=5)
    rows.append({
        "model": f"{cfg.hidden_units}",
        "alpha": cfg.alpha,
        "lr": cfg.learning_rate_init,
        "max_iter": cfg.max_iter,
        "cv_auc_mean": res["auc_mean"],
        "cv_auc_std": res["auc_std"],
        "cv_ap_mean": res["ap_mean"],
        "cv_f1_mean": res["f1_mean"],
        "cv_best_threshold": res["best_threshold"],
        "config": cfg,
        "result": res,
    })

tuning_table = pd.DataFrame(rows).sort_values(["cv_auc_mean", "cv_ap_mean", "cv_f1_mean"], ascending=False)
display(tuning_table[["model", "alpha", "lr", "max_iter", "cv_auc_mean", "cv_auc_std", "cv_ap_mean", "cv_f1_mean", "cv_best_threshold"]].round(4))

best_idx = tuning_table.index[0]
best_config = tuning_table.loc[best_idx, "config"]
best_result = tuning_table.loc[best_idx, "result"]
best_threshold = float(best_result["best_threshold"])

print("Best config:", best_config)
print("Chosen threshold:", round(best_threshold, 3))


## 5) Final model on held-out test set

After selecting the best CV configuration, I retrain on all training data and evaluate once on untouched test data.


In [ ]:
pre = build_preprocessor(categorical_cols, numeric_cols)
X_train_p = pre.fit_transform(X_train).astype(np.float32)
X_test_p = pre.transform(X_test).astype(np.float32)

final_model = build_model(best_config)
final_model.fit(X_train_p, y_train)

y_prob = final_model.predict_proba(X_test_p)[:, 1]
y_pred = (y_prob >= best_threshold).astype(int)

metrics = {
    "ROC-AUC": roc_auc_score(y_test, y_prob),
    "PR-AUC": average_precision_score(y_test, y_prob),
    "F1": f1_score(y_test, y_pred, zero_division=0),
    "Precision": precision_score(y_test, y_pred, zero_division=0),
    "Recall": recall_score(y_test, y_pred, zero_division=0),
    "Accuracy": accuracy_score(y_test, y_pred),
}

pd.DataFrame([metrics]).round(4)


## 6) Confusion matrix and clinical interpretation

Confusion matrix entries:
- **TN**: correctly predicted non-recurrence
- **FP**: predicted recurrence but actually non-recurrence
- **FN**: predicted non-recurrence but actually recurrence (**most critical error here**)
- **TP**: correctly predicted recurrence

For this problem, reducing FN is important because missing recurrence risk can have a high clinical cost.


In [ ]:
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=["Pred: No recurrence", "Pred: Recurrence"],
            yticklabels=["Actual: No recurrence", "Actual: Recurrence"])
plt.title("Confusion Matrix (Test Set)")
plt.tight_layout()
plt.show()

print(f"TN={tn}, FP={fp}, FN={fn}, TP={tp}")
print("\nClassification report:")
print(classification_report(y_test, y_pred, target_names=["no-recurrence", "recurrence"], zero_division=0))


## 7) So what? (conclusion)

1. The model is tuned systematically (multiple architectures + regularization + learning rates + CV).
2. Performance is evaluated on metrics beyond accuracy (ROC-AUC, PR-AUC, F1, precision, recall).
3. The decision threshold is tuned for better F1 and can be lowered further if the priority is to reduce false negatives.
4. The confusion matrix explicitly quantifies FN, which is critical for recurrence-risk screening use.

This makes the workflow traceable, reproducible, and clinically interpretable.


## 8) Save reusable outputs


In [ ]:
joblib.dump(
    {"preprocessor": pre, "model": final_model, "threshold": best_threshold},
    "breast_cancer_recurrence_model.joblib",
)
pd.Series(y_prob, name="recurrence_probability").to_csv("test_recurrence_probabilities.csv", index=False)

print("Saved model bundle: breast_cancer_recurrence_model.joblib")
print("Saved probabilities: test_recurrence_probabilities.csv")
